In [2]:
'''

We perform two calibration processes:
    
    One long calibration, where we calibrate with
        Co-60, Na (preamp 100)
        Ba, Cs, Co-57 (preamp 0; Co-57 done overnight)
    this long calibration is done only once (Feb 24; Co-57 run was done 5:45 PM Feb 24-3PM Feb 25).

    *Every* time we take data, we will do a single Cs (preamp level 0) and a Co-60 (preamp level 100) run to 
    see the variation in our ADC output compared to the previous baseline.
    
'''

'\n\nWe perform two calibration processes:\n\n    One long calibration, where we calibrate with\n        Co-60, Na (preamp 100)\n        Ba, Cs, Co-57 (preamp 0; Co-57 done overnight)\n    this long calibration is done only once (Feb 24; Co-57 run was done 5:45 PM Feb 24-3PM Feb 25).\n\n    *Every* time we take data, we will do a single Cs run to see the variation in our ADC output compared to the previous baseline.\n\n'

In [3]:
'''

Naming schema: notation.

I denote file folders with [] and nuclear data files with {}

'''

'\n\nNaming schema: notation.\n\nI denote file folders with [] and nuclear data files with {}\n\n'

In [4]:
'''

ROUGH FILE STRUCTURE

[] calibration
    [] feb_24
        [] preamp_100
            {} Co60 _22.5C_preamp100_feb24
            {} Na_22.5C_preamp100_feb24
        [] preamp_0
            {} Ba_22.4C_preamp0_feb24
            {} Cs_22.1C_preamp0_feb24
    [] daily
        {} Cs_22.4C_preamp0_feb26
        {} Co60_22.4C_preamp100_feb26

'''

'\n\nROUGH FILE STRUCTURE\n\n[] calibration\n    [] feb_24\n        [] preamp_100\n            {} Co60 _22.5C_preamp100_feb24\n            {} Na_22.5C_preamp100_feb24\n        [] preamp_0\n            {} Ba_22.4C_preamp0_feb24\n            {} Cs_22.1C_preamp0_feb24\n    [] daily\n        {} Cs_22.4C_preamp0_feb24\n\n'

In [ ]:
'''

Goal:
    Generate a calibration curve from the big run (Feb 24)
        There should be two such curves, one for preamp level 0 and preamp level 100
    
    Generate a data algorithm which adjusts our day-by-day analysis for changes in ADC output
        There should be two such curves, one for preamp level 0 and preamp level 100
    Any change from the big run should be roughly on the order of 13 ADC channels (quite small, but worth accounting for anyways)
    We wantto do this by forcefully changing the slope (?) or something from the original calibration curve
    
'''

In [ ]:
# -----------------------------
# IMPORT STATEMENTS
# -----------------------------

# --- computer imports ---
from pathlib import Path
import os
import re
import pandas as pd

# --- data analysis imports ---
import matplotlib.pyplot as plt

import numpy as np
from numpy import genfromtxt

from scipy.special import factorial
from scipy.stats import poisson, binomtest, norm
from scipy.stats import chi2 as chi2_dist
from scipy.optimize import curve_fit
from scipy.special import erf

In [ ]:
# -----------------------------
# INPUTS: KNOWN SPECTROSCOPY DATA
# -----------------------------

'''
Using Idaho National Labratory's NaI Gamma spec data

All listed values are in keV
'''

# -----------------------------
# PREAMP 0 
# -----------------------------

# --- used for daily calibration curve ---
# see https://gammaray.inl.gov/SiteAssets/catalogs/nai/pdf/cs137.pdf
Cs_spectra     = np.array{31.8, 37.3, 661.657} 
Cs_spectra_err = np.array{0,    0,    0.003}
# ---                                  ---

# see https://gammaray.inl.gov/SiteAssets/catalogs/nai/pdf/ba133.pdf
Ba_spectra     = np.array{53.15, 79.60, 80.998, 160.605, 223.246, 276.397, 302.851, 356.005, 383.851}
Ba_spectra_err = np.array{0.05,  0.05,  0.008,  0.015,   0.030,   0.012,   0.015,   0.017,   0.020}



# -----------------------------
# PREAMP 100 
# -----------------------------

# --- used for daily calibration curve ---
# see https://gammaray.inl.gov/SiteAssets/catalogs/nai/pdf/co60.pdf
Co60_spectra     = np.array{1173.228, 1332.494}
Co60_spectra_err = np.array{0.003,    0.002}
# ---                                  ---

# see https://gammaray.inl.gov/SiteAssets/catalogs/nai/pdf/na22.pdf
Na_spectra     = np.array{511.006, 1274.537}
Na_spectra_err = np.array{0,       0.008}


# -----------------------------
# OVERNIGHT RUNS: Co-57 (set to preamp 0) 
# -----------------------------

# see https://gammaray.inl.gov/SiteAssets/catalogs/nai/pdf/co57.pdf
Co57_spectra     = np.array{14.4136, 122.060, 136.471, 570.1, 692.4}
Co57_spectra_err = np.array{0.0003,  0.010,   0.010,   0.2,   0.1}

In [ ]:
# -----------------------------
# HELPER FUNCTIONS: generate initial calibration curve
# -----------------------------

In [ ]:
# -----------------------------
# HELPER FUNCTIONS: generate calibration curve for the day
# -----------------------------